# Lecturer voice — Piper (VITS), the *fast* one

The **realtime** voice for the Telegram call bot. XTTS/F5 are the quality voices;
Piper is non-autoregressive (RTF ≪ 1), ~tens-of-ms latency. Notes: `kaggle_finetune_piper.md`.

Training stack = **`OHF-Voice/piper1-gpl`** (maintained Piper).

### ⚠️ Cell order matters: data first, piper install last
`datasets`/`pandas` need **numpy 2** (Kaggle's image); piper1-gpl's training deps
pin **numpy <2**. They can't share one kernel. So we **pull + prep the data on the
stock image first**, write wavs to disk, and only **then install piper** — its
numpy downgrade is harmless after that (training runs in a `!python` subprocess;
upload uses pure-python `huggingface_hub`). Do **not** reorder these cells.

### Storage: HuggingFace only
No Kaggle Datasets/output, no gdrive. data in ← two HF datasets · model out →
`HF_MODEL_REPO` · resume ↔ `resume.ckpt` there. local `/tmp/voice` scratch is throwaway,
rebuilt from HF each run.

**Before running:** Accelerator → **GPU**; Secrets → `HF_TOKEN` (*write*).

## CONFIG

In [ ]:
# ============================== CONFIG ==============================
import os
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ['HF_TOKEN']
os.environ['HF_TOKEN'] = HF_TOKEN

HF_USER       = 'cyttic'
HF_DATASETS   = ['cyttic/audio-kri-russian', 'cyttic/audio-kri-russian-2']
HF_MODEL_REPO = f'{HF_USER}/lecturer-ru-piper'   # voice + resume.ckpt live here
RESUME_NAME   = 'resume.ckpt'
VOICE_NAME    = 'lecturer_ru'

BASE_REPO = 'rhasspy/piper-checkpoints'
BASE_DIR  = 'ru/ru_RU/irina/medium'                  # female base ~ female target
BASE_CKPT = f'{BASE_DIR}/epoch=4139-step=929464.ckpt'    # irina/medium (verified)

ESPEAK_VOICE = 'ru'
SR           = 22050        # medium quality
MAX_EPOCHS   = 4640        # ABSOLUTE; base irina=4139, so ~500 epochs (~3h). 6000 would be ~11h/overfit. Raise+re-run to train more.
BATCH        = 16
WORK   = '/tmp/voice'      # local scratch only; NOT a Kaggle dir
os.makedirs(WORK, exist_ok=True)
DATA   = f'{WORK}/ds'; WAVS = f'{DATA}/wav'; META = f'{DATA}/metadata.csv'
CACHE  = f'{WORK}/cache'; CONFIG = f'{WORK}/config.json'
PRE    = f'{WORK}/pretrained'
# ===================================================================


## 1 · Pull both HF datasets → `wav/` + `metadata.csv`  (stock numpy-2 env)

Runs **before** piper is installed. Raw WAV bytes decoded with `soundfile`
(`Audio(decode=False)`, no torchcodec), resampled to `SR` mono 16-bit.
CSV is piper1-gpl format: `clip.wav|text`.

In [ ]:
import io, csv, soundfile as sf, librosa
from datasets import load_dataset, Audio
os.makedirs(WAVS, exist_ok=True)
n = 0
with open(META, 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f, delimiter='|')
    for repo in HF_DATASETS:
        ds = load_dataset(repo, split='train', token=HF_TOKEN).cast_column('audio', Audio(decode=False))
        for r in ds:
            text = (r.get('text') or '').strip()
            if not text: continue
            a, sr = sf.read(io.BytesIO(r['audio']['bytes']), dtype='float32', always_2d=True)
            a = a.mean(axis=1)
            if sr != SR: a = librosa.resample(a, orig_sr=sr, target_sr=SR)
            n += 1
            fn = f'clip_{n:04d}.wav'
            sf.write(f'{WAVS}/{fn}', a, SR, subtype='PCM_16')
            w.writerow([fn, text])
print(f'{n} clips -> {WAVS} ({SR} Hz mono 16-bit)')
print(open(META, encoding='utf-8').read()[:300])


### 1b · Clean Whisper hallucination rows

In [ ]:
BAD = ['продолжение следует', 'субтитры', 'редактор субтитров',
       'amara.org', 'подпишись', 'спасибо за просмотр']
kept = []
for line in open(META, encoding='utf-8').read().splitlines():
    if '|' not in line: continue
    fn, text = line.split('|', 1)
    t = text.lower().strip()
    if len(t) < 3 or any(b in t for b in BAD):
        p = f'{WAVS}/{fn}'
        if os.path.exists(p): os.remove(p)
        continue
    kept.append((fn, text))
with open(META, 'w', encoding='utf-8', newline='') as f:
    csv.writer(f, delimiter='|').writerows(kept)
print(f'{len(kept)} clips after cleaning')


## 2 · Starting checkpoint — resume from HF, else base  (still stock env)

`huggingface_hub` is pure-python, so this works before *or* after the piper
install; we do it now while the env is clean.

In [ ]:
from huggingface_hub import hf_hub_download, HfApi
os.makedirs(PRE, exist_ok=True)
api = HfApi(token=HF_TOKEN)
have = False
try:    have = RESUME_NAME in api.list_repo_files(HF_MODEL_REPO, repo_type='model')
except Exception: pass
if have:
    START_CKPT = hf_hub_download(HF_MODEL_REPO, RESUME_NAME, local_dir=PRE, token=HF_TOKEN)
    print('RESUMING from HF:', START_CKPT)
else:
    START_CKPT = hf_hub_download(BASE_REPO, BASE_CKPT, repo_type='dataset', local_dir=PRE)
    print('FIRST RUN, base:', START_CKPT)
# other RU bases (repo_type='dataset'): denis/dmitri/ruslan /medium; list:
#   print([f for f in api.list_repo_files(BASE_REPO, repo_type='dataset') if BASE_DIR in f])


## 3 · Install piper1-gpl  ← only now (downgrades numpy; data is already on disk)

Builds the **espeakbridge** C extension, which CMake-downloads & compiles
espeak-ng (`CMakeLists` needs cmake ≥ 3.26 → we pip-install a recent one, since
`setup.py build_ext` runs un-isolated). The cell **gates on `from piper import
espeakbridge`** — if that line fails, read the CMake output above it.

After this cell `datasets`/`pandas` are broken in-kernel — expected; everything
below uses `!python` subprocesses or pure-python `huggingface_hub`.

In [ ]:
PIPER = f'{WORK}/piper1-gpl'
# espeakbridge is a C extension that downloads + compiles espeak-ng via CMake.
# CMakeLists needs cmake >= 3.26 and scikit-build; Kaggle's apt cmake is older,
# so install a recent cmake into THIS env (setup.py build_ext runs un-isolated).
!apt-get -qq install -y build-essential ninja-build git python3-dev espeak-ng >/dev/null
!pip -q install 'scikit-build' 'cmake>=3.26' ninja
![ -d {PIPER} ] || git clone -q https://github.com/OHF-voice/piper1-gpl.git {PIPER}
%cd {PIPER}
!cmake --version | head -1
!pip -q install -e '.[train]'
!pip -q install onnxscript          # torch.onnx.export (PyTorch 2.6) needs it
!./build_monotonic_align.sh
# Full output on purpose: if espeak-ng/espeakbridge fails to build, the CMake
# error shows here (don't pipe to tail).
!python setup.py build_ext --inplace
%cd {WORK}
# Hard gate: the import that failed before must now work.
!python -c "from piper import espeakbridge; import piper.train; print('espeakbridge + piper.train OK')"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 3b · Sanitize the base checkpoint (drop stale hparams)

Lightning replays the checkpoint's saved `hyper_parameters` through the arg
parser when `--ckpt_path` is set. Old rhasspy checkpoints carry keys the current
`VitsModel` doesn't accept (e.g. `sample_bytes`) → `fit` aborts with *"does not
accept option 'model.sample_bytes' … Parsing of ckpt_path hyperparameters
failed!"*. We keep only hparams in `VitsModel`'s signature and re-save; the model
is still built from the CLI `--model.*` args. No-op for our own `resume.ckpt`.

In [ ]:
import sys, os, torch, inspect
_src = f'{WORK}/piper1-gpl/src'
if os.path.isdir(_src) and _src not in sys.path:
    sys.path.insert(0, _src)
try:
    from piper.train.vits.lightning import VitsModel
except ModuleNotFoundError as e:
    raise SystemExit('piper not installed in THIS kernel — run §3 first.') from e

valid = set(inspect.signature(VitsModel.__init__).parameters) - {'self'}
ck = torch.load(START_CKPT, map_location='cpu', weights_only=False)
hp = dict(ck.get('hyper_parameters') or {})
stale = [k for k in hp if k not in valid]
for k in stale: hp.pop(k, None)
ck['hyper_parameters'] = hp
CLEAN = os.path.join(PRE, 'start_clean.ckpt')
torch.save(ck, CLEAN)
START_CKPT = CLEAN
print('dropped stale hparams:', stale)

# PyTorch 2.6 loads ckpts with weights_only=True; the old checkpoint stores
# pathlib.Path objects it then refuses to unpickle. Launch piper.train via this
# wrapper so those classes are allow-listed BEFORE Lightning loads the ckpt.
RUN = f'{WORK}/run_piper.py'
open(RUN, 'w').write(
    'import sys, torch.serialization, pathlib, runpy\n'
    'torch.serialization.add_safe_globals([pathlib.PosixPath, pathlib.PurePosixPath,\n'
    '    pathlib.WindowsPath, pathlib.PureWindowsPath, pathlib.PurePath])\n'
    'runpy.run_module(sys.argv.pop(1), run_name="__main__")\n')
print('start:', START_CKPT, '| launcher:', RUN)


## 4 · Fine-tune

`fit` folds phonemize/caching in. `--ckpt_path` loads the start checkpoint and
the **epoch counter continues from it** (irina = 4139), so `MAX_EPOCHS` is
absolute. ~11 steps/epoch, ~21 s/epoch on a T4 → keep it short: ~500 epochs
(`MAX_EPOCHS≈4640`, ~3 h) for a first audition; 6000 = ~11 h and overfits 15 min
of audio.

Checkpoints save every 10 epochs, so you can **interrupt once the §6 samples sound
good** (after ≥10 epochs there's a saved ckpt for §5 to export). To train more
later, raise `MAX_EPOCHS` and re-run the notebook — it resumes from HF `resume.ckpt`.

In [ ]:
os.makedirs(CACHE, exist_ok=True)
!python {RUN} piper.train fit \
  --data.voice_name {VOICE_NAME} \
  --data.csv_path {META} \
  --data.audio_dir {WAVS} \
  --data.espeak_voice {ESPEAK_VOICE} \
  --data.cache_dir {CACHE} \
  --data.config_path {CONFIG} \
  --data.batch_size {BATCH} \
  --model.sample_rate {SR} \
  --trainer.accelerator gpu --trainer.devices 1 \
  --trainer.max_epochs {MAX_EPOCHS} \
  --trainer.precision 32 \
  --ckpt_path {START_CKPT}
# CUDA OOM? lower --data.batch_size (8/4). Only MEDIUM base ckpts load cleanly.


## 5 · Export to ONNX (+ config → `.onnx.json`)

In [ ]:
import glob, os, shutil, subprocess
subprocess.run(['pip','install','-q','onnxscript'])   # torch.onnx needs it (safety)
# Force the legacy TorchScript ONNX exporter: recent torch defaults to the new
# dynamo exporter, which fails on VITS's data-dependent spline
# (GuardOnDataDependentSymNode). piper's call omits dynamo=, so we add it.
_exp = f'{WORK}/piper1-gpl/src/piper/train/export_onnx.py'
_s = open(_exp).read()
if 'dynamo=' not in _s:
    _s = _s.replace('model=model_g,', 'model=model_g, dynamo=False,', 1)
    open(_exp, 'w').write(_s)
print('legacy exporter forced:', 'dynamo=False' in open(_exp).read())
# Find any training checkpoint under WORK (robust to the exact logger path),
# excluding the base/clean ckpts in pretrained/.
cands = [c for c in glob.glob(f'{WORK}/**/*.ckpt', recursive=True) if '/pretrained/' not in c]
cands = sorted(cands, key=os.path.getmtime)
print('training checkpoints found:', cands or '(none)')
assert cands, ('no training checkpoint under ' + WORK + ' — run §4 (fit) in THIS '
               'session and let it save at least one (every 10 epochs) before '
               'exporting. A previous session\'s ckpts are wiped on a new kernel.')
LAST_CKPT = cands[-1]
print('exporting from:', LAST_CKPT)
VOICE = f'{WORK}/{VOICE_NAME}.onnx'   # local scratch file; uploaded to HF in §6
# ONNX export is CPU-only; hide the GPU so a full/busy GPU can't OOM the
# CUDA init (export needs no GPU).
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': ''}
r = subprocess.run(['python', RUN, 'piper.train.export_onnx',
                    '--checkpoint', LAST_CKPT, '--output-file', VOICE],
                   capture_output=True, text=True, env=env)
print(r.stdout[-2000:]);  print(r.stderr[-3000:])
assert r.returncode == 0 and os.path.isfile(VOICE), (
    'export_onnx FAILED (rc=%d).\n=== stderr tail ===\n%s\n=== stdout tail ===\n%s'
    % (r.returncode, r.stderr[-1800:], r.stdout[-800:]))
shutil.copy(CONFIG, VOICE + '.json')
print('exported OK ->', VOICE, '(+ .json),', os.path.getsize(VOICE)//1024, 'KB')


## 6 · Save to HuggingFace (voice **and** resume checkpoint)

In [ ]:
api.create_repo(HF_MODEL_REPO, repo_type='model', private=True, exist_ok=True)
api.upload_file(path_or_fileobj=VOICE,           path_in_repo=f'{VOICE_NAME}.onnx',      repo_id=HF_MODEL_REPO)
api.upload_file(path_or_fileobj=VOICE + '.json', path_in_repo=f'{VOICE_NAME}.onnx.json', repo_id=HF_MODEL_REPO)
api.upload_file(path_or_fileobj=LAST_CKPT,       path_in_repo=RESUME_NAME,               repo_id=HF_MODEL_REPO)
print('saved ->', f'https://huggingface.co/{HF_MODEL_REPO}')
print('serve: put the two .onnx files in voice-agent/voices/ and set VA_TTS_VOICE')


## 7 · Quick listen

In [ ]:
TEXT = 'Привет! Это тест быстрого клонированного голоса лектора.'
!echo "{TEXT}" | piper -m {VOICE} -f {WORK}/test.wav
from IPython.display import Audio
Audio(f'{WORK}/test.wav')
